# VideoPrism: Fine-tuning for Video Classification with a Frozen Backbone

[![Paper](https://img.shields.io/badge/arXiv-2402.13217-red.svg)](https://arxiv.org/abs/2402.13217)
[![Blog](https://img.shields.io/badge/Google_Research-Blog-green.svg)](https://research.google/blog/videoprism-a-foundational-visual-encoder-for-video-understanding/)
[![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)

This notebook shows how to fine-tune **VideoPrism** for video classification ([`FactorizedVideoClassifier`](../encoders.py#L583)) by keeping the
pre-trained backbone ([`FactorizedEncoder`](../encoders.py#L583)) frozen and training only a lightweight attention-pooler ([`AttenTokenPoolingLayer`](../encoders.py#L619)) + projection head ([`FeedForward`](../encoders.py#L647)).


We walk through **two complete examples** that cover different use-cases:

| Example | Dataset | Task | Loss | Evaluation |
|---|---|---|---|---|
| **1** | CalMS21 Task 1 | Multi-label animal behaviour | Sigmoid BCE | mAP, Mean Class Acc |
| **2** | Kinetics-400 | Single-label action recognition | Softmax CE | Top-1, Top-5, Mean Class Acc |

Both examples use **the same shared functions** — they differ only in configuration.

---

Data download instructions are provided in each example section. A small subsample of the data is
included for testing. Place downloaded data in the `vc_data` folder (or using soft link) to match the paths, or update the absolute paths in accordingly.

```
vc_data/
├── CalMS21/
│   └── train/
│   └── test/
├── k400/
│   ├── annotations/
│   ├── test/
│   └── train/
└── k400_examples/
```



> **Tip:** Run this notebook on Google Colab with a GPU or TPU runtime for best performance.

---
## 1. Environment Setup

In [ ]:
# @title Prepare environment

import os, sys, subprocess

# Fetch VideoPrism repository if Python does not know about it and install
# dependencies needed for this notebook.
if not os.path.exists("videoprism_repo"):
  !git clone --quiet --branch=main --depth=1 \
     https://github.com/google-deepmind/videoprism.git videoprism_repo
  os.chdir('./videoprism_repo')
  # If tensorflow is installed, remove the last line of requirements.txt to avoid version conflict
  result = subprocess.run(['pip', 'show', 'tensorflow'], capture_output=True)
  if result.returncode == 0:
    !sed -i '$d' requirements.txt
  !pip install .
  os.chdir('..')

# Append VideoPrism code to Python import path.
if "videoprism_repo" not in sys.path:
  sys.path.append("videoprism_repo")

# Install missing dependencies.
!pip install mediapy
!pip install jax

import jax
import tensorflow as tf
from jax.extend import backend

# Prevent TF from claiming GPU/TPU devices — JAX owns them.
tf.config.set_visible_devices([], "GPU")
tf.config.set_visible_devices([], "TPU")

print(f"JAX version:  {jax.__version__}")
print(f"JAX platform: {backend.get_backend().platform}")
print(f"JAX devices:  {jax.device_count()}")

In [ ]:
# @title Download vc_data from GitHub

import os

if not os.path.exists("vc_data"):
    os.symlink("videoprism_repo/videoprism/colabs/vc_data", "vc_data")
    print("vc_data linked from cloned repo.")
else:
    print("vc_data already exists, skipping.")

---
## 2. Shared Functions

All cells in this section are dataset-agnostic. Each function accepts `multilabel` and
`num_classes` arguments so the same code drives both examples below.

In [ ]:
import datetime
import functools
import glob
import json
import math
import os
import pickle
import random
from concurrent.futures import ThreadPoolExecutor

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mediapy
import numpy as np
import optax
import PIL.Image
import tensorflow as tf
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

from videoprism import models as vp

NUM_FRAMES = 16
FRAME_SIZE = 180

In [ ]:
# @title Model and optimizer setup

MODEL_NAME = "videoprism_public_v1_base"  # @param ["videoprism_public_v1_base", "videoprism_public_v1_large"]


def build_model_and_params(num_classes: int):
    """Creates FactorizedVideoClassifier and injects pretrained encoder weights.

    Returns params with three top-level keys:
        'encoder'      — frozen pretrained FactorizedEncoder
        'atten_pooler' — randomly initialised attention pooler (will be trained)
        'projection'   — randomly initialised linear head (will be trained)

    Args:
        num_classes: Output dimension of the classification head.

    Returns:
        (classifier, params)
    """
    classifier = vp.videoprism_vc_v1_base(num_classes=num_classes)
    key = jax.random.PRNGKey(0)
    dummy = jnp.zeros((1, NUM_FRAMES, FRAME_SIZE, FRAME_SIZE, 3))
    variables = classifier.init(key, dummy, train=False)
    pretrained = vp.load_pretrained_weights(MODEL_NAME)
    params = dict(variables["params"])
    params["encoder"] = pretrained["params"]
    print("Parameter subtrees:", list(params.keys()))
    # Expected: ['encoder', 'atten_pooler', 'projection']
    return classifier, params


def build_optimizer(params: dict, learning_rate: float = 1e-4):
    """Partitioned optimizer: encoder uses set_to_zero (frozen), head uses Adam.

    Args:
        params: Parameter dict from build_model_and_params.
        learning_rate: Adam learning rate for the trainable head.

    Returns:
        (tx, opt_state)
    """
    def _tag(subtree, label: str):
        return jax.tree_util.tree_map(lambda _: label, subtree)

    param_labels = {
        k: _tag(v, "frozen" if k == "encoder" else "trainable")
        for k, v in params.items()
    }
    tx = optax.multi_transform(
        {
            "trainable": optax.adam(learning_rate=learning_rate),
            "frozen":    optax.set_to_zero(),
        },
        param_labels,
    )
    opt_state = tx.init(params)
    print("Optimizer ready — encoder: frozen | atten_pooler + projection: trainable")
    return tx, opt_state

In [ ]:
# @title Training and evaluation step factories

def make_train_step(classifier, tx, multilabel: bool = False):
    """Returns a JIT-compiled training step function.

    Single-label (multilabel=False): softmax cross-entropy with integer labels.
    Multi-label  (multilabel=True):  sigmoid binary cross-entropy with multi-hot labels.
    Accuracy is always reported as top-1 (argmax) for a comparable training signal.

    Args:
        classifier: The Flax classifier module.
        tx: Optax optimizer.
        multilabel: Whether to use sigmoid BCE instead of softmax CE.

    Returns:
        train_step(params, opt_state, batch_videos, batch_labels)
            -> (new_params, new_opt_state, loss, accuracy)
    """
    if multilabel:
        def _loss_and_acc(logits, batch_labels):
            loss = optax.sigmoid_binary_cross_entropy(logits, batch_labels).mean()
            pred = jnp.argmax(logits, axis=-1)
            true = jnp.argmax(batch_labels, axis=-1)
            return loss, jnp.mean(pred == true)
    else:
        def _loss_and_acc(logits, batch_labels):
            loss = optax.softmax_cross_entropy_with_integer_labels(logits, batch_labels).mean()
            return loss, jnp.mean(jnp.argmax(logits, axis=-1) == batch_labels)

    @jax.jit
    def train_step(params, opt_state, batch_videos, batch_labels):
        def loss_fn(p):
            logits, _ = classifier.apply({"params": p}, batch_videos, train=True)
            loss, _ = _loss_and_acc(logits, batch_labels)
            return loss, logits
        (loss, logits), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        updates, new_opt_state = tx.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        _, acc = _loss_and_acc(logits, batch_labels)
        return new_params, new_opt_state, loss, acc

    return train_step


def make_eval_step(classifier):
    """Returns a JIT-compiled evaluation step (inference only, train=False).

    Returns:
        eval_step(params, batch_videos) -> logits [B, num_classes]
    """
    @jax.jit
    def eval_step(params, batch_videos):
        logits, _ = classifier.apply({"params": params}, batch_videos, train=False)
        return logits
    return eval_step

In [ ]:
def compute_metrics(all_logits: np.ndarray, all_labels: np.ndarray,
                    multilabel: bool = False, ignore_background: bool = False) -> dict:
    """Computes evaluation metrics appropriate for the task.

    Single-label (multilabel=False):
        Top-1, Top-5, and Mean Class Accuracy.
        all_labels should be int32 [N].

    Multi-label (multilabel=True):
        mAP (macro-averaged AP over classes present in the eval set) and
        Mean Class Accuracy (per-class binary accuracy averaged over classes).
        all_labels should be float32 [N, C] multi-hot.

    Args:
        all_logits: float32 [N, num_classes] — raw model outputs.
        all_labels: int32 [N] or float32 [N, C] ground-truth labels.
        multilabel: Switch between single- and multi-label metric sets.
        ignore_background: If True, excludes the background class (assumed to be the last class) from all metrics.

    Returns:
        Dict of metric names to float values in [0, 1].
    """
    num_classes = all_logits.shape[-1]
    if ignore_background:
        bg_class = num_classes - 1
        all_logits = all_logits[:, :-1]
        if multilabel:
            all_labels = all_labels[:, :-1]
        else:
            mask = all_labels != bg_class
            all_logits = all_logits[mask]
            all_labels = all_labels[mask]
        num_classes -= 1

    if multilabel:
        present = all_labels.sum(axis=0) > 0  # classes with ≥1 positive sample
        map_score = float(average_precision_score(
            all_labels[:, present], all_logits[:, present], average="macro"
        ))
        preds = (all_logits > 0).astype(np.float32)  # threshold at logit 0 ↔ sigmoid > 0.5
        mean_class_acc = float(np.mean(np.mean(preds == all_labels, axis=0)))
        return dict(map=map_score, mean_class_acc=mean_class_acc)
    else:
        preds_top1 = np.argmax(all_logits, axis=-1)
        top1 = float(np.mean(preds_top1 == all_labels))
        # argpartition is O(N) vs O(N log N) for argsort — faster for large num_classes.
        preds_top5 = np.argpartition(all_logits, -5, axis=-1)[:, -5:]
        top5 = float(np.mean([lbl in row for lbl, row in zip(all_labels, preds_top5)]))
        correct = np.zeros(num_classes, dtype=np.float32)
        total   = np.zeros(num_classes, dtype=np.float32)
        for lbl, pred in zip(all_labels, preds_top1):
            total[lbl]   += 1
            correct[lbl] += int(lbl == pred)
        present = total > 0
        mean_class_acc = float(np.mean(correct[present] / total[present]))
        return dict(top1=top1, top5=top5, mean_class_acc=mean_class_acc)


def evaluate(eval_step_fn, params, batches, n_batches: int,
             multilabel: bool = False, ignore_background: bool = False) -> dict:
    """Runs full evaluation over a labeled video dataset.

    Args:
        eval_step_fn: JIT-compiled inference function from make_eval_step.
        params: Current model parameters.
        batches: Iterable of (batch_videos, batch_labels) from make_batch.
        n_batches: Total number of batches (used for the progress bar total).
        multilabel: Whether labels are multi-hot (True) or integer (False).
        ignore_background: If True, excludes the background class (assumed to be the last class) from all metrics.

    Returns:
        Dict of evaluation metrics.
    """
    all_logits, all_labels = [], []

    pbar = tqdm(batches, total=n_batches, desc="Evaluating", unit="batch")
    for batch_videos, batch_labels in pbar:
        logits = eval_step_fn(params, jnp.asarray(batch_videos))
        all_logits.append(np.asarray(logits))
        all_labels.append(batch_labels)
        pbar.set_postfix(samples=sum(len(l) for l in all_labels))

    metrics = compute_metrics(
        np.concatenate(all_logits, axis=0),
        np.concatenate(all_labels, axis=0),
        multilabel=multilabel,
        ignore_background=ignore_background,
    )
    if multilabel:
        print(f"  mAP            : {metrics['map']*100:.2f}%")
        print(f"  Mean Class Acc : {metrics['mean_class_acc']*100:.2f}%")
    else:
        print(f"  Top-1  Acc     : {metrics['top1']*100:.2f}%")
        print(f"  Top-5  Acc     : {metrics['top5']*100:.2f}%")
        print(f"  Mean Class Acc : {metrics['mean_class_acc']*100:.2f}%")
    return metrics

In [ ]:
# @title Checkpointing helpers

def save_checkpoint(ckpt_dir: str, params, opt_state, epoch: int, global_step: int) -> str:
    """Serialises params and optimizer state to disk. Returns the saved path."""
    os.makedirs(ckpt_dir, exist_ok=True)
    path = os.path.join(ckpt_dir, f"checkpoint_step{global_step:07d}.pkl")
    state = {
        "params":      jax.tree_util.tree_map(np.array, params),
        "opt_state":   jax.tree_util.tree_map(np.array, opt_state),
        "epoch":       epoch,
        "global_step": global_step,
    }
    with open(path, "wb") as f:
        pickle.dump(state, f)
    return path


def load_checkpoint(path: str) -> dict:
    """Loads a checkpoint saved by save_checkpoint."""
    with open(path, "rb") as f:
        return pickle.load(f)

In [ ]:
# @title Training loop

def train(
    classifier,
    params,
    tx,
    opt_state,
    train_batches,
    n_train_batches: int,
    val_batches=None,
    n_val_batches: int | None = None,
    num_epochs: int = 50,
    multilabel: bool = False,
    ckpt_dir: str = "checkpoints/finetune",
    ckpt_every: int = 20,
    keep_recent: int = 5,
    resume_ckpt_dir: str | None = None,
):
    """Runs the full training loop with checkpointing.

    Args:
        classifier: The Flax classifier module.
        params: Initial model parameters (ignored when resuming).
        tx: Optax optimizer.
        opt_state: Initial optimizer state (ignored when resuming).
        train_batches: Iterable of (batch_videos, batch_labels) from make_batch.
        n_train_batches: Total number of training batches (used for the progress bar total).
        val_batches: Optional iterable of (batch_videos, batch_labels) for per-epoch validation.
            Pass None to skip validation.
        n_val_batches: Total number of validation batches (used for the progress bar total).
        num_epochs: Total number of training epochs.
        multilabel: Use sigmoid BCE and multi-hot labels when True.
        ckpt_dir: Parent directory for checkpoint files.
        ckpt_every: Save a checkpoint every N steps.
        keep_recent: Number of most-recent checkpoints to keep on disk.
        resume_ckpt_dir: Path to a run's checkpoint directory to resume from.
            If provided, the most recent checkpoint is loaded and training
            continues from the next epoch.

    Returns:
        (params, opt_state, history)
    """
    if resume_ckpt_dir is not None:
        ckpt_files = sorted(glob.glob(os.path.join(resume_ckpt_dir, "checkpoint_step*.pkl")))
        if not ckpt_files:
            raise FileNotFoundError(f"No checkpoints found in {resume_ckpt_dir}")
        state        = load_checkpoint(ckpt_files[-1])
        params       = state["params"]
        opt_state    = state["opt_state"]
        start_epoch  = state["epoch"] + 1
        global_step  = state["global_step"]
        run_id       = os.path.basename(resume_ckpt_dir)
        run_ckpt_dir = resume_ckpt_dir
        recent_ckpts = list(ckpt_files[-keep_recent:])
        print(f"Resuming run {run_id} from epoch {state['epoch']} (step {global_step})")
    else:
        run_id       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        run_ckpt_dir = os.path.join(ckpt_dir, run_id)
        start_epoch  = 1
        global_step  = 0
        recent_ckpts = []
        print(f"Run ID: {run_id}")

    train_step = make_train_step(classifier, tx, multilabel=multilabel)
    eval_step  = make_eval_step(classifier)
    primary_key = "map" if multilabel else "top1"
    history     = dict(loss=[], acc=[], val_primary=[])

    best_val_primary = -1.0
    best_ckpt_path: str | None = None
    last_ckpt_path: str | None = None

    for epoch in tqdm(range(start_epoch, num_epochs + 1), desc="Epochs", unit="epoch"):
        loss, acc = None, None
        batch_bar = tqdm(
            train_batches,
            total=n_train_batches,
            desc=f"  Epoch {epoch}/{num_epochs}",
            unit="batch",
            leave=False,
        )
        for batch_videos, batch_labels in batch_bar:
            params, opt_state, loss, acc = train_step(
                params, opt_state, jnp.asarray(batch_videos), jnp.asarray(batch_labels),
            )
            global_step += 1

            if global_step % ckpt_every == 0:
                last_ckpt_path = save_checkpoint(run_ckpt_dir, params, opt_state, epoch, global_step)
                recent_ckpts.append(last_ckpt_path)
                if len(recent_ckpts) > keep_recent:
                    evicted = recent_ckpts.pop(0)
                    if evicted != best_ckpt_path:
                        os.remove(evicted)

        epoch_loss, epoch_acc = float(loss), float(acc)
        history["loss"].append(epoch_loss)
        history["acc"].append(epoch_acc)

        log_str = f"Epoch {epoch:>3}/{num_epochs} — loss: {epoch_loss:.4f}, acc: {epoch_acc:.4f}"

        if val_batches is not None:
            val_metrics = evaluate(eval_step, params, val_batches, n_val_batches,
                                   multilabel=multilabel)
            primary_val = val_metrics[primary_key]
            history["val_primary"].append(primary_val)
            log_str += f", val_{primary_key}: {primary_val:.4f}"
            if primary_val > best_val_primary and last_ckpt_path is not None:
                best_val_primary = primary_val
                best_ckpt_path   = last_ckpt_path
                print(f"  New best checkpoint (val_{primary_key}={best_val_primary:.4f}): {best_ckpt_path}")

        print(log_str)

    return params, opt_state, history


def plot_training_curves(history: dict, val_metric_name: str = "Val metric"):
    """Plots loss and accuracy / validation curves from a training history dict."""
    epochs = range(1, len(history["loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle("Training Curves", fontsize=13, fontweight="bold")

    ax1.plot(epochs, history["loss"], marker="o", linewidth=2, label="Train loss")
    ax1.set(xlabel="Epoch", ylabel="Loss", title="Loss")
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend()

    ax2.plot(epochs, [a * 100 for a in history["acc"]], marker="o", linewidth=2,
             color="tab:orange", label="Train acc")
    if history.get("val_primary"):
        ax2.plot(epochs, [v * 100 for v in history["val_primary"]], marker="s",
                 linewidth=2, linestyle="--", color="tab:green", label=val_metric_name)
    ax2.set(xlabel="Epoch", ylabel="(%)", title="Accuracy")
    ax2.set_ylim(0, 105)
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()

    plt.tight_layout()
    plt.show()


---
## Example 1: CalMS21 Task 1 — Multi-Label Animal Behaviour Recognition

[CalMS21](https://data.caltech.edu/records/s0vdx-0k302) (Caltech Mouse Social Interactions
Dataset, 2021) is a benchmark for automated animal behaviour analysis from video. Task 1
focuses on classifying social interactions between pairs of mice filmed from above.

**Task:** Given a 16-frame clip of two interacting mice, predict a multi-hot label vector over
4 behaviour classes: `{attack, investigation, mount, other}`.

**Metrics:**
* **mAP** (mean Average Precision) — the primary CalMS21 metric; averaged over classes that
  have at least one positive example in the evaluation set.
* **Mean Class Accuracy** — per-class binary accuracy (threshold logits at 0, i.e. sigmoid > 0.5),
  averaged over all classes.

Class index mapping:

| Index | Behaviour |
|---|---|
| 0 | attack |
| 1 | investigation |
| 2 | mount |
| 3 | other |

**Data Preparation**
* This example uses a subsample of the processed data from [scivid](https://storage.googleapis.com/scivid).  
* The full dataset can be found in the [scivid cloud storage bucket](https://storage.googleapis.com/scivid).

In [ ]:
# @title Dataset preparation

# download a subset of the training set for the few-shot example

!mkdir -p vc_data/CalMS21/train
!bash -c 'gsutil ls "gs://scivid/full/calms21_slim/calms21_train_split_ids_shuffle_scale05_downsample16.array_record-*" \
| head -n 20 \
| xargs -I {} gsutil cp {} vc_data/CalMS21/train'

!mkdir -p vc_data/CalMS21/test
!bash -c 'gsutil ls "gs://scivid/full/calms21_slim/calms21_test_split_ids_shuffle_scale05.array_record-*" \
| head -n 10  \
| xargs -I {} gsutil cp {} vc_data/CalMS21/test'

In [ ]:
# @title Example 1 — Configuration

CALMS_NUM_CLASSES   = 4    # attack, investigation, mount, other
CALMS_NUM_EPOCHS    = 10
CALMS_BATCH_SIZE    = 40  # lower if running out of memory
CALMS_LEARNING_RATE = 5e-5
CALMS_CKPT_EVERY    = 20
CALMS_KEEP_RECENT   = 5
CALMS_MULTILABEL    = True   # multi-label → sigmoid BCE + multi-hot labels
CALMS_CKPT_DIR   = "checkpoints/calms21_finetune"

CALMS_CLASS_NAMES = ["attack", "investigation", "mount", "other"]
CALMS_IGNORE_BG = True  # exclude the "other" background class for evaluation



In [ ]:
# @title Example 1 — Create shots and batches

import os
import glob
import random
import collections
import numpy as np
import tensorflow as tf
from array_record.python.array_record_data_source import ArrayRecordDataSource



In [ ]:
CTX_FEAT = {"clip/label/index": tf.io.VarLenFeature(dtype=tf.int64)}
SEQ_FEAT = {"image/encoded": tf.io.FixedLenSequenceFeature((), dtype=tf.string)}

def create_shots(
    path: str,
    split: str,
    n_shots: int,
) -> dict[int, list[bytes]]:
    """Scan shards matching `path/split` and collect up to n_shots raw records per class.

    Parameters
    ----------
    path:    directory containing the array_record shards.
    split:   glob pattern for shards relative to `path` (e.g. "calms21_train_*.array_record-*").
    n_shots: number of examples to collect per class.

    Returns
    -------
    dict mapping class index -> list of raw SequenceExample bytes.
    """
    def _get_label(raw: bytes) -> int:
        ctx, _, _ = tf.io.parse_sequence_example(raw, CTX_FEAT, {})
        return int(tf.sparse.to_dense(ctx["clip/label/index"]).numpy().flat[0])

    shard_paths = sorted(glob.glob(os.path.join(path, split)))
    collected: dict[int, list[bytes]] = collections.defaultdict(list)
    for shard_path in shard_paths:
        source = ArrayRecordDataSource([shard_path])
        for i in range(len(source)):
            raw = source[i]
            label = _get_label(raw)
            if len(collected[label]) < n_shots:
                collected[label].append(raw)

        # Check if we're done
        if all(len(v) >= n_shots for v in collected.values()):
            print("Done collecting!")
            break

    return dict(collected)

In [ ]:
def decode_record(raw: bytes, frame_size: int | None = None) -> tuple[np.ndarray, int]:
    """Decode a raw SequenceExample -> (frames float32 [T,H,W,3], label int)."""
    ctx, seq, _ = tf.io.parse_sequence_example(raw, CTX_FEAT, SEQ_FEAT)
    label = int(tf.sparse.to_dense(ctx["clip/label/index"]).numpy().flat[0])
    frames = []
    for f in seq["image/encoded"]:
        frame = tf.image.decode_jpeg(f, channels=3)
        if frame_size is not None:
            h, w = tf.shape(frame)[0], tf.shape(frame)[1]
            scale = tf.cast(frame_size, tf.float64) / tf.cast(tf.minimum(h, w), tf.float64)
            new_h = tf.cast(tf.round(tf.cast(h, tf.float64) * scale), tf.int32)
            new_w = tf.cast(tf.round(tf.cast(w, tf.float64) * scale), tf.int32)
            frame = tf.image.resize(frame, [new_h, new_w])
            frame = tf.image.resize_with_crop_or_pad(frame, frame_size, frame_size)
        frames.append(frame.numpy())
    frames = np.stack(frames).astype(np.float32) / 255.0  # [T, H, W, 3]
    return frames, label


def make_batch(
    collected: dict[int, list[bytes]],
    batch_size: int = 4,
    one_hot: bool = False,
    frame_size: int | None = None,
    n_batches: int = 1,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Sample `batch_size` records uniformly from `collected`, yielding `n_batches` times.

    Returns
    -------
    videos : float32 [B, T, H, W, 3]
    labels : int32   [B]        (one_hot=False)
             float32 [B, C]     (one_hot=True)
    """
    all_raw = [(label, raw) for label, raws in collected.items() for raw in raws]
    n_classes = max(collected.keys()) + 1

    for _ in range(n_batches):
        sampled = random.sample(all_raw, batch_size)

        frames_list, labels_list = [], []
        for label, raw in sampled:
            frames, _ = decode_record(raw, frame_size=frame_size)
            frames_list.append(frames)
            labels_list.append(label)

        videos = np.stack(frames_list).astype(np.float32)  # [B, T, H, W, 3]
        labels = np.array(labels_list, dtype=np.int32)      # [B]

        if one_hot:
            labels = np.eye(n_classes, dtype=np.float32)[labels]  # [B, C]

        yield videos, labels

In [ ]:
# @title Example 1 — Train

calms_classifier, calms_params = build_model_and_params(num_classes=CALMS_NUM_CLASSES)
calms_tx, calms_opt_state = build_optimizer(calms_params, learning_rate=CALMS_LEARNING_RATE)


DATA_DIR = "vc_data/CalMS21/train"
SPLIT = "calms21_train_split_ids_shuffle_scale05_downsample16.array_record-*-of-01024"

# create 10 shots per class for small-scale demonstration
# in practice, you might want to use more shots and/or the full training set
collected = create_shots(DATA_DIR, SPLIT, n_shots=10)
calms_n_train_batches = math.ceil(sum(len(v) for v in collected.values()) / CALMS_BATCH_SIZE)
# Convert to list so the batches can be re-iterated across epochs.
# Fresh random samples are drawn each time make_batch is called.
calms_train_batches = list(make_batch(
    collected, batch_size=CALMS_BATCH_SIZE, one_hot=True,
    frame_size=FRAME_SIZE, n_batches=calms_n_train_batches,
))

calms_params, calms_opt_state, calms_history = train(
    calms_classifier, calms_params, calms_tx, calms_opt_state,
    train_batches=calms_train_batches,
    n_train_batches=calms_n_train_batches,
    num_epochs=CALMS_NUM_EPOCHS,
    multilabel=CALMS_MULTILABEL,
    ckpt_dir=CALMS_CKPT_DIR,
    ckpt_every=CALMS_CKPT_EVERY,
    keep_recent=CALMS_KEEP_RECENT,
)

In [ ]:
# @title Example 1 — Plot training curves

plot_training_curves(calms_history, val_metric_name="Val mAP")


In [ ]:
# @title Example 1 — Evaluate

DATA_DIR = "vc_data/CalMS21/test"
SPLIT = "calms21_test_split_ids_shuffle_scale05.array_record-*-of-01024"

# create 100 shots per class for small-scale evaluation
collected = create_shots(DATA_DIR, SPLIT, n_shots=100)
calms_n_test_batches = math.ceil(sum(len(v) for v in collected.values()) / CALMS_BATCH_SIZE)

# Convert to list so the batches can be re-iterated across epochs.
# Fresh random samples are drawn each time make_batch is called.
calms_test_batches = list(make_batch(
    collected, batch_size=CALMS_BATCH_SIZE, one_hot=True,
    frame_size=FRAME_SIZE, n_batches=calms_n_test_batches,
))

calms_eval_step  = make_eval_step(calms_classifier)
print("CalMS21 Task 1 test-set metrics:")
calms_metrics = evaluate(
    calms_eval_step, calms_params, calms_test_batches, calms_n_test_batches,
    multilabel=CALMS_MULTILABEL,
    ignore_background=CALMS_IGNORE_BG,
)

In [ ]:
# @title Example 1 — Per-class AP breakdown

# Recompute per-class AP from the test set (ignore the background)
all_logits, all_labels = [], []
for batch_videos, batch_labels in tqdm(
    calms_test_batches,
    total=calms_n_test_batches, desc="Collecting logits",
):
    all_logits.append(np.asarray(calms_eval_step(calms_params, jnp.asarray(batch_videos))))
    all_labels.append(batch_labels)

all_logits = np.concatenate(all_logits, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

per_class_ap = [
    float(average_precision_score(all_labels[:, c], all_logits[:, c]))
    if all_labels[:, c].sum() > 0 else float("nan")
    for c in range(CALMS_NUM_CLASSES - 1)
]

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#e07070", "#70a8e0", "#70c870"]
bars = ax.bar(CALMS_CLASS_NAMES[:-1], [v * 100 for v in per_class_ap],
              color=colors, edgecolor="k", linewidth=0.8)
ax.axhline(calms_metrics["map"] * 100, color="black", linestyle="--",
           linewidth=1.5, label=f"mAP = {calms_metrics['map']*100:.1f}%")
for bar, v in zip(bars, per_class_ap):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f"{v*100:.1f}%", ha="center", va="bottom", fontsize=10)
ax.set(xlabel="Behaviour", ylabel="Average Precision (%)",
       title="CalMS21 Task 1 — Per-class AP", ylim=(0, 110))
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## Example 2: Kinetics-400 — Single-Label Action Recognition

[Kinetics-400](https://github.com/cvdfoundation/kinetics-dataset) is the standard benchmark for
video action recognition. It contains ~240k training clips (10-second YouTube snippets) across
400 human-action classes such as *abseiling*, *playing guitar*, and *swimming*.

**Task:** Given a 16-frame clip, predict a **single** class label from 400 categories.  
**Loss:** Softmax cross-entropy with integer labels (`multilabel=False`).  
**Metrics:** Top-1, Top-5, Mean Class Accuracy.

### Dataset Preparation:

Download the dataset following: https://github.com/cvdfoundation/kinetics-dataset

soft link downloaded and extracted dataset to the vc_data folder

### Dataset json file format for input

```json
[
  {"file": "/data/k400/abseiling/clip_0001.mp4",      "label_idx": 0},
  {"file": "/data/k400/guitar_playing/clip_0123.mp4", "label_idx": 152},
  ...
]
```

> **10-shot setting:** `k400_train_10shot.json` contains 10 randomly sampled clips per class
> (4 000 clips total) — useful for data-efficient evaluation.

In [ ]:
# @title Example 2 — Data loading and preprocessing functions for K400

def read_and_preprocess_frames(
    source,
    target_num_frames: int = NUM_FRAMES,
    target_frame_size: tuple = (FRAME_SIZE, FRAME_SIZE),
) -> np.ndarray:
    """Loads a video clip or image sequence and returns float32 [T, H, W, 3].

    Args:
        source: A video file path (str) or a list of image file paths (list[str]).
        target_num_frames: Number of frames to uniformly sample.
        target_frame_size: (height, width) after resize + center-crop.

    Returns:
        float32 array [T, H, W, 3] with pixel values in [0, 1].
    """
    if isinstance(source, list):
        indices = np.linspace(0, len(source), num=target_num_frames, endpoint=False, dtype=np.int32)
        frames = np.stack([
            np.array(PIL.Image.open(source[i]).convert("RGB")) for i in indices
        ])
    else:
        frames = mediapy.read_video(source)
        indices = np.linspace(0, len(frames), num=target_num_frames, endpoint=False, dtype=np.int32)
        frames = np.asarray([frames[i] for i in indices])

    h, w = frames.shape[1], frames.shape[2]
    target_h, target_w = target_frame_size
    scale = max(target_h / h, target_w / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    if (new_h, new_w) != (h, w):
        frames = mediapy.resize_video(frames, shape=(new_h, new_w))
    top  = (new_h - target_h) // 2
    left = (new_w - target_w) // 2
    frames = frames[:, top:top + target_h, left:left + target_w]
    return mediapy.to_float01(frames)


def load_pairs(json_path: str) -> list:
    """Reads a JSON dataset file and returns (source, label_idx) pairs.

    Supported entry formats:
        Single-label video:  {"file": "/path/clip.mp4", "label_idx": 3}
        Multi-label images:  {"files": ["/path/f0.png", ...], "label_idx": [0, 2]}
    """
    with open(json_path) as f:
        data = json.load(f)
    return [
        (entry["file"] if "file" in entry else entry["files"], entry["label_idx"])
        for entry in data
    ]


def _encode_labels(batch_labels: list, multilabel: bool, num_classes: int) -> np.ndarray:
    """Encodes a list of label values.

    Single-label: returns int32 [B].
    Multi-label:  returns float32 [B, num_classes] multi-hot vectors.
    """
    if multilabel:
        arr = np.zeros((len(batch_labels), num_classes), dtype=np.float32)
        for i, idxs in enumerate(batch_labels):
            for idx in idxs:
                arr[i, idx] = 1.0
        return arr
    return np.array(batch_labels, dtype=np.int32)


def make_batch(
    file_label_pairs,
    batch_size: int = 4,
    num_workers: int = 4,
    shuffle: bool = False,
    drop_remainder: bool = False,
    multilabel: bool = False,
    num_classes: int = 400,
):
    """Yields (videos, labels) batches with parallel video I/O.

    Args:
        file_label_pairs: Iterable of (source, label_idx) tuples, where source
            is a video path (str) or a list of image paths (list[str]).
        batch_size: Number of videos per batch.
        num_workers: Threads for parallel frame loading.
        shuffle: Shuffle dataset order each epoch.
        drop_remainder: Drop the last incomplete batch.
        multilabel: Encode labels as float32 multi-hot [B, C].
        num_classes: Total number of classes; used only when multilabel=True.

    Yields:
        (videos, labels): float32 [B, T, H, W, 3] and int32 [B] or float32 [B, C].
    """
    pairs = list(file_label_pairs)
    if shuffle:
        random.shuffle(pairs)

    def _load(source_label):
        source, label = source_label
        return read_and_preprocess_frames(source), label

    batch_videos, batch_labels = [], []
    chunk_size = num_workers * 2
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        for start in range(0, len(pairs), chunk_size):
            chunk = pairs[start : start + chunk_size]
            for frames, label in pool.map(_load, chunk):
                batch_videos.append(frames)
                batch_labels.append(label)
                if len(batch_videos) == batch_size:
                    yield np.stack(batch_videos), _encode_labels(batch_labels, multilabel, num_classes)
                    batch_videos, batch_labels = [], []
    if batch_videos and not drop_remainder:
        yield np.stack(batch_videos), _encode_labels(batch_labels, multilabel, num_classes)


In [ ]:
# @title Example 2 — Configuration

K400_NUM_CLASSES   = 400
K400_NUM_EPOCHS    = 5
K400_BATCH_SIZE    = 16
K400_LEARNING_RATE = 1e-4
K400_CKPT_EVERY    = 20
K400_KEEP_RECENT   = 5
K400_MULTILABEL    = False   # single-label → softmax CE + integer labels

# Adjust these paths to point to your own dataset files.
K400_TRAIN_JSON = "vc_data/k400_examples/k400_train_10shot.json"   # 10-shot split (4 000 clips)
K400_TEST_JSON  = "vc_data/k400_examples/k400_test_10shot.json"
K400_CKPT_DIR   = "checkpoints/k400_finetune"


In [ ]:
# @title Example 2 — Train

k400_classifier, k400_params = build_model_and_params(num_classes=K400_NUM_CLASSES)
k400_tx, k400_opt_state = build_optimizer(k400_params, learning_rate=K400_LEARNING_RATE)

k400_train_pairs = load_pairs(K400_TRAIN_JSON)
k400_n_train_batches = math.ceil(len(k400_train_pairs) / K400_BATCH_SIZE)
k400_train_batches = list(make_batch(k400_train_pairs, batch_size=K400_BATCH_SIZE, shuffle=True,
                                multilabel=K400_MULTILABEL, num_classes=K400_NUM_CLASSES))

k400_params, k400_opt_state, k400_history = train(
    k400_classifier, k400_params, k400_tx, k400_opt_state,
    train_batches=k400_train_batches,
    n_train_batches=k400_n_train_batches,
    num_epochs=K400_NUM_EPOCHS,
    multilabel=K400_MULTILABEL,
    ckpt_dir=K400_CKPT_DIR,
    ckpt_every=K400_CKPT_EVERY,
    keep_recent=K400_KEEP_RECENT,
)


In [ ]:
# @title Example 2 — Plot the training curves
plot_training_curves(k400_history, val_metric_name="Val Top-1")

In [ ]:
# @title Example 2 — Evaluate
k400_eval_step = make_eval_step(k400_classifier)
k400_test_pairs = load_pairs(K400_TEST_JSON)
k400_n_batches = math.ceil(len(k400_test_pairs) / K400_BATCH_SIZE)
k400_batches = list(make_batch(k400_test_pairs, batch_size=K400_BATCH_SIZE,
                          multilabel=K400_MULTILABEL, num_classes=K400_NUM_CLASSES))
print("K400 test-set metrics:")
k400_metrics = evaluate(
    k400_eval_step, k400_params, k400_batches, k400_n_batches,
    multilabel=K400_MULTILABEL,
)